In [0]:
from pyspark.sql import functions as F

mappings = {
    "alpha_medical": {
        "invoicedate": "sale_date", "customername": "hospital_name_raw",
        "itemcode": "product_code", "qty": "quantity", "netamountjpy": "amount_jpy"},
    "kanto_pharma": {
        "sales_dt": "sale_date", "ship_to_name": "hospital_name_raw",
        "product_cd": "product_code", "units": "quantity", "sales_amt": "amount_jpy"},
    "sakura_medlink": {
        "transaction_date": "sale_date", "facility": "hospital_name_raw",
        "sku": "product_code", "quantity": "quantity", "amount_jpy": "amount_jpy"},
}

In [0]:
frames = []
for dist, cols in mappings.items():
    bronze = spark.table(f"dbw_dp750_anshul.pharma.bronze_{dist}")
    df = bronze.select(
        *[F.col(src).alias(tgt) for src, tgt in cols.items()],
        F.lit(dist).alias("distributor"),
        "source_file", "ingested_at")
    frames.append(df)

conformed = frames[0]
for df in frames[1:]:
    conformed = conformed.unionByName(df)

conformed.write.mode("overwrite").saveAsTable("dbw_dp750_anshul.pharma.silver_sales_conformed")
display(conformed.groupBy("distributor").count())

In [0]:
%sql
SELECT distributor, sale_date, hospital_name_raw, product_code, quantity, amount_jpy
FROM dbw_dp750_anshul.pharma.silver_sales_conformed
ORDER BY RAND()
LIMIT 15;

In [0]:
from pyspark.sql import functions as F

date_formats = {
    "alpha_medical":  "yyyy-MM-dd",
    "kanto_pharma":   "dd/MM/yyyy",
    "sakura_medlink": "yyyy/MM/dd",
}

# build: WHEN distributor = alpha THEN parse with alpha's format, WHEN kanto ... etc.
date_expr = None
for dist, fmt in date_formats.items():
    condition = F.col("distributor") == dist
    parsed = F.try_to_timestamp(F.col("sale_date_raw"), F.lit(fmt))
    if date_expr is None:
        date_expr = F.when(condition, parsed)
    else:
        date_expr = date_expr.when(condition, parsed)

conformed = spark.table("dbw_dp750_anshul.pharma.silver_sales_conformed")

typed = (conformed
    .withColumnRenamed("sale_date", "sale_date_raw")                 # keep the original text
    .withColumn("sale_date", date_expr.cast("date"))                 # real DATE (NULL if invalid)
    .withColumn("quantity", F.col("quantity").try_cast("int"))
    .withColumn("amount_jpy", F.col("amount_jpy").try_cast("bigint")))

typed.write.mode("overwrite").saveAsTable("dbw_dp750_anshul.pharma.silver_sales_typed")
typed.printSchema()

In [0]:
%sql
SELECT distributor,
       COUNT(*)                                          AS total_rows,
       SUM(CASE WHEN sale_date IS NULL THEN 1 ELSE 0 END) AS invalid_dates,
       SUM(CASE WHEN quantity  IS NULL THEN 1 ELSE 0 END) AS invalid_qty,
       SUM(CASE WHEN quantity < 0      THEN 1 ELSE 0 END) AS negative_qty,
       MIN(sale_date) AS first_date,
       MAX(sale_date) AS last_date
FROM dbw_dp750_anshul.pharma.silver_sales_typed
GROUP BY distributor
ORDER BY distributor;

In [0]:
from pyspark.sql import functions as F

typed = spark.table("dbw_dp750_anshul.pharma.silver_sales_typed")

clean = (typed
    # 1. hospital name: trim ends, collapse inner spaces, upper case
    .withColumn("hospital_name_std",
        F.upper(F.regexp_replace(F.trim(F.col("hospital_name_raw")), r"\s+", " ")))
    # 2. product code: trim + upper; empty text -> NULL
    .withColumn("product_code",
        F.nullif(F.upper(F.trim(F.col("product_code"))), F.lit("")))
    # 3. business flag
    .withColumn("is_return", F.col("quantity") < 0)
    # 4. data-quality flags
    .withColumn("dq_invalid_date",    F.col("sale_date").isNull())
    .withColumn("dq_missing_product", F.col("product_code").isNull()))

clean.write.mode("overwrite").saveAsTable("dbw_dp750_anshul.pharma.silver_sales_clean")
display(clean.select("distributor", "hospital_name_raw", "hospital_name_std",
                     "product_code", "is_return", "dq_invalid_date").limit(10))

In [0]:
%sql
SELECT distributor,
       COUNT(DISTINCT hospital_name_raw)                AS distinct_raw,
       COUNT(DISTINCT hospital_name_std)                AS distinct_std,
       SUM(CASE WHEN dq_missing_product THEN 1 ELSE 0 END) AS missing_product,
       SUM(CASE WHEN is_return          THEN 1 ELSE 0 END) AS returns,
       SUM(CASE WHEN dq_invalid_date    THEN 1 ELSE 0 END) AS invalid_dates
FROM dbw_dp750_anshul.pharma.silver_sales_clean
GROUP BY distributor
ORDER BY distributor;

In [0]:
from pyspark.sql import functions as F

clean = spark.table("dbw_dp750_anshul.pharma.silver_sales_clean")

business_key = ["distributor", "sale_date_raw", "hospital_name_raw",
                "product_code", "quantity", "amount_jpy"]

dedup = clean.dropDuplicates(business_key)

before = clean.count()
after = dedup.count()
print(f"before: {before}, after: {after}, removed: {before - after}")

dedup.write.mode("overwrite").saveAsTable("dbw_dp750_anshul.pharma.silver_sales")

In [0]:
%sql
-- 1) rows per distributor after dedup
SELECT distributor, COUNT(*) AS rows
FROM dbw_dp750_anshul.pharma.silver_sales
GROUP BY distributor ORDER BY distributor;

In [0]:
%sql
-- 2) prove no duplicates remain (should return 0 rows)
SELECT distributor, sale_date_raw, hospital_name_raw, product_code, quantity, amount_jpy,
       COUNT(*) AS n
FROM dbw_dp750_anshul.pharma.silver_sales
GROUP BY ALL
HAVING COUNT(*) > 1;